In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))

In [ ]:
# Library imports
import numpy as np
import pandas as pd
import time
import os

# File imports
from env import InventoryEnv
from miscelaneous import read_instance
from policies.parameterized_lookahead_approximation_policy import ParameterizedLookaheadApproximationPolicy

In [ ]:
def run_episode(env, policy, instance_file):

    instance = read_instance(instance_file)
    env.set_instance(instance)
    policy.reset(instance) 
    state, info = env.reset()

    done = False
    final_reward = 0

    while not done:
        action = policy.act(state)
        state, reward, done, truncated, _ = env.step(action)
        final_reward += reward

    return final_reward

#Evaluate the policy
def evaluate_policy(env, policy, instances):

    list_final_reward = [run_episode(env, policy, instance_file) for instance_file in instances]

    return np.mean(list_final_reward), np.std(list_final_reward), list_final_reward

In [ ]:
def get_cfa_params(num_warehouses, num_customers, capacity_distribution):

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution, grid_size=200)

    # Disjoint from export_results.ipynb's test set (instances_seed_10001..10200), so the
    # param chosen here isn't validated on the same instances it'll later be graded on.
    num_instances = 50
    instances = [f'../instances/instances_train_parameterized_lookahead_approximation/instances_seed_{i}.json' for i in range(30001, 30001 + num_instances)]

    params = [1.00, 1.02, 1.04, 1.06, 1.08, 1.10]

    all_results = []
    best_param = None
    best_mean_reward = -np.inf
    best_std_reward = None

    for param in params:

        policy = ParameterizedLookaheadApproximationPolicy(env, num_warehouses, num_customers, capacity_distribution, param=param)
        mean_reward, std_reward, all_rewards = evaluate_policy(env, policy, instances)

        all_results.append({
            'num_warehouses': num_warehouses,
            'num_customers': num_customers,
            'capacity_distribution': capacity_distribution,
            'param': param,
            'mean_reward': round(mean_reward, 2),
            'std_reward': round(std_reward, 2),
            'all_rewards': all_rewards
        })

        if mean_reward > best_mean_reward:
            best_mean_reward = mean_reward
            best_param = param
            best_std_reward = std_reward

    # Store every candidate param's reward, so the sweep can be inspected/plotted later
    all_results_df = pd.DataFrame(all_results)
    all_results_csv_path = 'parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_all_params.csv'
    all_results_df.to_csv(all_results_csv_path, mode='a', header=not os.path.exists(all_results_csv_path), index=False)

    # Store the winning param (plus its own reward) for ParameterizedLookaheadPolicy to look up
    best_result_df = pd.DataFrame([{
        'num_warehouses': num_warehouses,
        'num_customers': num_customers,
        'capacity_distribution': capacity_distribution,
        'best_param': best_param,
        'mean_reward': round(best_mean_reward, 2),
        'std_reward': round(best_std_reward, 2)
    }])
    
    best_csv_path = 'parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_best_params.csv'
    best_result_df.to_csv(best_csv_path, mode='a', header=not os.path.exists(best_csv_path), index=False)

In [5]:
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform', 'uneven']

training_times = []  # one row per (num_warehouses, num_customers, capacity_distribution) family

for num_warehouses in num_warehouses_options:
    for num_customers in num_customers_options:
        for capacity_distribution in capacity_distribution_options:
            print(f"Running experiments for {num_warehouses} warehouses, {num_customers} customers and {capacity_distribution} capacity distribution")

            start_time = time.perf_counter()
            get_cfa_params(num_warehouses, num_customers, capacity_distribution)
            training_time_minutes = (time.perf_counter() - start_time) / 60

            training_times.append({
                'num_warehouses': num_warehouses,
                'num_customers': num_customers,
                'capacity_distribution': capacity_distribution,
                'training_time': round(training_time_minutes, 2),
            })

training_times_df = pd.DataFrame(training_times)
training_times_csv_path = 'parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_training_times.csv'
training_times_df.to_csv(training_times_csv_path, index=False, float_format='%.5f')

Running experiments for 2 warehouses, 50 customers and uniform capacity distribution
Set parameter Username
Set parameter LicenseID to value 2776932
Academic license - for non-commercial use only - expires 2027-02-09
Running experiments for 2 warehouses, 50 customers and uneven capacity distribution
Running experiments for 2 warehouses, 100 customers and uniform capacity distribution
Running experiments for 2 warehouses, 100 customers and uneven capacity distribution
Running experiments for 2 warehouses, 200 customers and uniform capacity distribution
Running experiments for 2 warehouses, 200 customers and uneven capacity distribution
Running experiments for 2 warehouses, 400 customers and uniform capacity distribution
Running experiments for 2 warehouses, 400 customers and uneven capacity distribution
Running experiments for 3 warehouses, 50 customers and uniform capacity distribution
Running experiments for 3 warehouses, 50 customers and uneven capacity distribution
Running experimen